# 模型推理 - 使用 QLoRA 微调后的 ChatGLM4-9B

In [1]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HOME'] = '/hy-tmp'

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
import pandas as pd
import json

# 定义全局变量和参数
model_name_or_path = 'THUDM/glm-4-9b-chat-hf'  # 模型ID或本地路径
train_data_path = 'HasturOfficial/adgen'       # 训练数据路径
output_dir = "./model_output/chatglm4_adgen_lora"


/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1、加载base model

In [2]:
bnb_config = BitsAndBytesConfig(load_in_4bit = True,
                              bnb_4bit_quant_type = 'nf4',
                              bnb_4bit_use_double_quant = True,
                              bnb_4bit_compute_dtype = torch.bfloat16)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
base_model.requires_grad_(False)
base_model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:18<00:00,  4.68s/it]


GlmForCausalLM(
  (model): GlmModel(
    (embed_tokens): Embedding(151552, 4096, padding_idx=151329)
    (layers): ModuleList(
      (0-39): 40 x GlmDecoderLayer(
        (self_attn): GlmAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=True)
          (k_proj): Linear4bit(in_features=4096, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=4096, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): GlmMLP(
          (gate_up_proj): Linear4bit(in_features=4096, out_features=27392, bias=False)
          (down_proj): Linear4bit(in_features=13696, out_features=4096, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): GlmRMSNorm((4096,), eps=1.5625e-07)
        (post_attention_layernorm): GlmRMSNorm((4096,), eps=1.5625e-07)
      )
    )
    (norm): GlmRMSNorm((4096,), eps=1.5625e-07)
    (rotary_emb): GlmRotaryEmbedding()
  )

In [3]:
print(base_model.model.layers[0].self_attn.q_proj)

Linear4bit(in_features=4096, out_features=4096, bias=True)


In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('THUDM/glm-4-9b-chat-hf')

## 2、加载微调的模型

In [5]:
from peft import PeftModel, PeftConfig

peft_model_path = "./model_output/chatglm4_adgen_lora/"
peft_after_model = PeftModel.from_pretrained(base_model, peft_model_path)

In [6]:
# peft_after_model.set_adapter("default")

In [7]:
peft_after_model.print_trainable_parameters()

trainable params: 0 || all params: 9,411,850,240 || trainable%: 0.0


In [8]:
print(base_model.model.layers[0].self_attn.q_proj)

lora.Linear4bit(
  (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=True)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.1, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=4096, out_features=4, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=4, out_features=4096, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
)


## 3、加载评估数据集

In [9]:
from datasets import load_dataset

dataset = load_dataset(train_data_path)
validation_dataset = dataset["validation"]

In [10]:
validation_dataset

Dataset({
    features: ['content', 'summary'],
    num_rows: 1070
})

In [11]:
validation_dataset[0]

{'content': '类型#上衣*材质#牛仔布*颜色#白色*风格#简约*图案#刺绣*衣样式#外套*衣款式#破洞',
 'summary': '简约而不简单的牛仔外套，白色的衣身十分百搭。衣身多处有做旧破洞设计，打破单调乏味，增加一丝造型看点。衣身后背处有趣味刺绣装饰，丰富层次感，彰显别样时尚。'}

## 4、抽样观察baseModel和peftModel输出结果

In [12]:
message = [
    {
        "role": "system",
        "content": "根据输入的关键词信息，写一段生动的电商广告文案。"
    },
    {
        "role": "user",
        "content": validation_dataset[0]['content']
    }
]
print("validation_dataset[0]", validation_dataset[0])
print("\nmessage", message)

inputs = tokenizer.apply_chat_template(
    message,
    return_tensors='pt',
    tokenize = True, 
    add_generation_prompt = True,
    return_dict=True,
).to('cuda')
input_len = inputs['input_ids'].shape[1]
generate_kwargs = {
    "input_ids": inputs['input_ids'],
    "attention_mask": inputs['attention_mask'],
    "max_new_tokens": 128,
    "do_sample": False,
}

peft_model_out = peft_after_model.generate(**generate_kwargs)
print("\npeft_model:",tokenizer.decode(peft_model_out[0][input_len:], skip_special_tokens=True))

## base_model得主动先把LORA部件给关闭
with peft_after_model.disable_adapter():
    base_model_out = base_model.generate(**generate_kwargs)
    print("\nbase_model:",tokenizer.decode(base_model_out[0][input_len:], skip_special_tokens=True))



validation_dataset[0] {'content': '类型#上衣*材质#牛仔布*颜色#白色*风格#简约*图案#刺绣*衣样式#外套*衣款式#破洞', 'summary': '简约而不简单的牛仔外套，白色的衣身十分百搭。衣身多处有做旧破洞设计，打破单调乏味，增加一丝造型看点。衣身后背处有趣味刺绣装饰，丰富层次感，彰显别样时尚。'}

message [{'role': 'system', 'content': '根据输入的关键词信息，写一段生动的电商广告文案。'}, {'role': 'user', 'content': '类型#上衣*材质#牛仔布*颜色#白色*风格#简约*图案#刺绣*衣样式#外套*衣款式#破洞'}]

peft_model: 
这款牛仔外套，简约的版型，不挑身材，不挑人穿。衣身破洞设计，个性时尚，彰显不羁的时尚感。衣身白色刺绣设计，精致美观，彰显品质。

base_model: 

🌟【新品上市】简约白牛仔外套🌟

👕【材质】精选优质牛仔布，柔软舒适，亲肤透气。

🌈【颜色】纯净的白色，简约而不简单，轻松驾驭各种场合。

🎨【风格】简约风格，经典不过时，让你轻松成为街头焦点。

💎【图案】精致刺绣工艺，细节之处彰显品质，让你穿上即是品味。

👕【衣样式】外套设计，轻松应对温差变化，让你时刻保持最佳状态。

👕【衣款式】破洞设计，时尚前卫，打破


## 5、采用DeepSeekV4模型评估微调结果和验证集的差异

### 对验证集批量预测

In [13]:
# 1. 确保 Tokenizer 配置正确
tokenizer.padding_side = 'left' 
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def batch_predict(peft_after_model, tokenizer, dataset, batch_size=4):
    results = []
    # 构造所有的 messages 文本
    all_prompts = []
    for item in dataset:
        msg = [
            {"role": "system", "content": "根据输入的关键词信息，写一段生动的电商广告文案。"},
            {"role": "user", "content": item['content']}
        ]
        # 使用 tokenize=False 拿到拼接后的文本
        all_prompts.append(tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True))
    print(f"一共有{len(all_prompts)}条验证集样本")
    
    # 分批处理
    for i in range(0, len(all_prompts), batch_size):
        batch_texts = all_prompts[i : i + batch_size]
        # 编码并 Padding, tokenizer可以接受batch list形式输入
        inputs = tokenizer(batch_texts, return_tensors='pt', padding = True).to('cuda')
        # 生成
        with torch.no_grad():
            outputs = peft_after_model.generate(
                **inputs,
                max_new_tokens = 256,
                do_sample = True,
                repetition_penalty = 1.1 
            )
        
        # 解码并提取回复部分
        for j, output in enumerate(outputs):
            # 获取输入序列的长度，用于截断，只保留模型生成的部分
            input_len = inputs['input_ids'][j].shape[0]
            decoded_text = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            
            # 保存结果（对应回原始数据集索引）
            curr_idx = i + j
            results.append({
                "peft_out": decoded_text.strip(), # 用 strip() 去掉你讨厌的 \n
                "validation_data": dataset[curr_idx]['summary'],
                "validation_content": dataset[curr_idx]['content']
            })
        print(f"已完成: {i + batch_size}/{len(all_prompts)}")
    return results

# 调用（L40S 建议尝试 batch_size=8 或 16）
validation_eval_result_list = batch_predict(peft_after_model, tokenizer, validation_dataset, batch_size = 128)
with open('validation_eval_result_list.json', 'w', encoding='utf-8') as f:
    json.dump(validation_eval_result_list, f)
    

一共有1070条验证集样本
已完成: 128/1070
已完成: 256/1070
已完成: 384/1070
已完成: 512/1070
已完成: 640/1070
已完成: 768/1070
已完成: 896/1070
已完成: 1024/1070
已完成: 1152/1070


In [14]:
pd.DataFrame(validation_eval_result_list).head()

,peft_out,validation_data,validation_content
0,BRAND家这款牛仔外套，以白色为主色调，看起来纯净简约百搭性强，搭配上个性的破洞装饰，显得...,简约而不简单的牛仔外套，白色的衣身十分百搭。衣身多处有做旧破洞设计，打破单调乏味，增加一丝造...,类型#上衣*材质#牛仔布*颜色#白色*风格#简约*图案#刺绣*衣样式#外套*衣款式#破洞
1,这款上衣采用简单的半高领的设计样式，既起到了保暖的效果，又不会显得很沉闷。同时带有别致的背带...,这款BRAND针织两件套连衣裙，简约的纯色半高领针织上衣，修饰着颈部线，尽显优雅气质。同时搭...,类型#裙*材质#针织*颜色#纯色*风格#复古*风格#文艺*风格#简约*图案#格子*图案#纯色...
2,一款<UNK>时尚的儿童卫衣；充满活力的色彩中装饰着个性可爱的卡通图案，既能彰显孩子娇小甜美...,嘻哈玩转童年，随时<UNK>，没错，出街还是要靠卫衣来装酷哦！时尚个性的连帽设计，率性有范还...,类型#上衣*风格#嘻哈*图案#卡通*图案#印花*图案#撞色*衣样式#卫衣*衣款式#连帽
3,纯色调的运用总能带来意想不到的惊喜，设计师采用优雅的<UNK>蓝渲染整款西裤，尽显成熟女性的...,裤子是简约大方的版型设计，带来一种极简主义风格而且不乏舒适优雅感，是衣橱必不可少的一件百搭单...,类型#裤*风格#英伦*风格#简约
4,时尚的高腰版型的半身裙穿在身上可以起到拉长腿长的作用，让你看起来更加的纤瘦苗条。舒适柔软的面...,这款来自梵凯的半身裙富有十足的设计感，采用了别致的不规则设计，凸显出时尚前卫的格调，再搭配俏...,类型#裙*裙下摆#弧形*裙腰型#高腰*裙长#半身裙*裙款式#不规则*裙款式#收腰


### 使用DeepSeek评估模型输出和标准文案的差异

In [ ]:
from openai import OpenAI
import time

client = OpenAI(
    api_key="sk-d39e9fdad7e44688bcec335791c5ada8", 
    base_url="https://api.deepseek.com"
)

def get_deepseek_score(peft_text, ground_truth, content):
    prompt = f"""
    你是一位专业的电商文案评测专家。请对比以下两段文案，并给出评价。
    
    【商品关键词】：{content}
    【参考答案（人工编写）】：{ground_truth}
    【待评测文案（AI生成）】：{peft_text}
    
    请从以下维度评估【待评测文案】：
    1. 准确性：是否包含了所有关键词？
    2. 生动性：语言是否吸引人，是否有购买欲望？
    3. 自然度：是否通顺，有没有复读或机械感？
    
    最后请给【待评测文案】打分（0-10分），并简要说明理由。
    请仅输出 JSON 格式结果，例如：{{"score": 8.5, "reason": "..."}}
    """
    
    try:
        response = client.chat.completions.create(
            model="deepseek-chat", # 评测通常用普通 chat 模型即可，省钱且快
            messages=[
                {"role": "system", "content": "你是一个严谨的文案评估专家，只输出JSON。"},
                {"role": "user", "content": prompt},
            ],
            response_format={ 'type': 'json_object' } # 强制要求返回 JSON
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {str(e)}")
        return json.dumps({'score':0, 'reason':'error没输出'})

# 抽样评测前 5 条（API 调用要花钱且耗时，先别全跑）
llm_judge_result_list = []
for i in range(len(validation_eval_result_list)):
    if i%100 == 3:
        time.sleep(3)
    try:
        res = validation_eval_result_list[i]
        score_json = get_deepseek_score(res['peft_out'], res['validation_data'], res['validation_content'])
        llm_judge_dict = res
        llm_judge_dict['llm_judge_score'] = json.loads(score_json)['score']
        llm_judge_dict['llm_judge_reason'] = json.loads(score_json)['reason']
        llm_judge_result_list.append(llm_judge_dict)
        print(f"评测样本打分DeepSeek已完成: {i}/{len(validation_eval_result_list)}")
        # print(f"样本 {i+1} 评测结果：\n{llm_judge_dict}\n")
    except:
        print(f"评测样本{i}跳过异常")

llm_judge_data_df = pd.DataFrame(llm_judge_result_list)
llm_judge_data_df.to_csv("llm_judge_data_df.csv")

In [35]:
llm_judge_data_df.head(10)

,peft_out,validation_data,validation_content,llm_judge_score,llm_judge_reason
0,BRAND家这款牛仔外套，以白色为主色调，看起来纯净简约百搭性强，搭配上个性的破洞装饰，显得...,简约而不简单的牛仔外套，白色的衣身十分百搭。衣身多处有做旧破洞设计，打破单调乏味，增加一丝造...,类型#上衣*材质#牛仔布*颜色#白色*风格#简约*图案#刺绣*衣样式#外套*衣款式#破洞,8.0,准确性：包含了所有关键词（上衣、牛仔布、白色、简约、刺绣、外套、破洞），但未明确提及‘风格#...
1,这款上衣采用简单的半高领的设计样式，既起到了保暖的效果，又不会显得很沉闷。同时带有别致的背带...,这款BRAND针织两件套连衣裙，简约的纯色半高领针织上衣，修饰着颈部线，尽显优雅气质。同时搭...,类型#裙*材质#针织*颜色#纯色*风格#复古*风格#文艺*风格#简约*图案#格子*图案#纯色...,6.5,准确性方面，虽然提及了大部分关键词但忽略了‘复古文艺风格’的整体感；生动性一般，语言较为平淡...
2,一款<UNK>时尚的儿童卫衣；充满活力的色彩中装饰着个性可爱的卡通图案，既能彰显孩子娇小甜美...,嘻哈玩转童年，随时<UNK>，没错，出街还是要靠卫衣来装酷哦！时尚个性的连帽设计，率性有范还...,类型#上衣*风格#嘻哈*图案#卡通*图案#印花*图案#撞色*衣样式#卫衣*衣款式#连帽,8.0,文案准确包含了所有关键词（连帽、卡通、印花、撞色、卫衣等），但在生动性和购买欲望上略逊于参考...
3,纯色调的运用总能带来意想不到的惊喜，设计师采用优雅的<UNK>蓝渲染整款西裤，尽显成熟女性的...,裤子是简约大方的版型设计，带来一种极简主义风格而且不乏舒适优雅感，是衣橱必不可少的一件百搭单...,类型#裤*风格#英伦*风格#简约,8.5,文案涵盖了所有关键词（裤、英伦、简约），准确度高；语言生动，如“纯色调”“优雅蓝”“双扣袢与...
4,时尚的高腰版型的半身裙穿在身上可以起到拉长腿长的作用，让你看起来更加的纤瘦苗条。舒适柔软的面...,这款来自梵凯的半身裙富有十足的设计感，采用了别致的不规则设计，凸显出时尚前卫的格调，再搭配俏...,类型#裙*裙下摆#弧形*裙腰型#高腰*裙长#半身裙*裙款式#不规则*裙款式#收腰,6.5,准确性方面，包含了所有关键词；生动性方面，语言平实但缺乏吸引力和购买欲望，未突出设计感和时尚...
5,这是一件时髦百搭的廓形<UNK>，穿着舒适又能很好的掩饰腿部缺陷，遮肉又显瘦；衣身前短后长的...,这件衬衫的款式非常的宽松，利落的线条可以很好的隐藏身材上的小缺点，穿在身上有着很好的显瘦效果...,类型#上衣*版型#宽松*版型#显瘦*图案#线条*衣样式#衬衫*衣袖型#泡泡袖*衣款式#抽绳,6.5,准确性方面，遗漏了‘线条’关键词，且‘廓形’与‘宽松’不完全对应；生动性较好，但‘掩饰腿部缺...
6,这款睡袍式的连体裙，看起来非常精致有格调。采用真丝面料加上大裙摆设计，给人带来丝丝女人味，穿...,宫廷风的甜美蕾丝设计，清醒的蕾丝拼缝处，刺绣定制的贝壳花边，增添了裙子的精致感觉。超大的裙摆...,类型#裙*材质#蕾丝*风格#宫廷*图案#刺绣*图案#蕾丝*裙型#大裙摆*裙下摆#花边*裙袖型...,4.5,准确性不足：未包含'蕾丝'和'宫廷'（虽提到'宫廷气质风范'，但核心面料蕾丝缺失），且出现<...
7,这款黑色紧身运动时尚九分裤，首先其采用修身的版型，让你拥有迷人的身材比例；其次这款裤子使用经...,个性化的九分裤型，穿着在身上，能够从视觉上拉长你的身体比例，让你看起来更加的有范。简约的黑色...,类型#裤*版型#显瘦*颜色#黑色*风格#简约*裤长#九分裤,6.0,准确性上，文案涵盖了‘版型显瘦’、‘黑色’、‘简约’、‘九分裤’等关键词，但未明确提及‘黑色...
8,"经典又十分百搭的廓形连衣裙，简约不失设计感,上身很是显瘦。大圆领的设计，修饰颈部曲线更显柔美...",文艺个性的印花连衣裙，藏青色底蕴，低调又大气，撞色太阳花分布整个裙身，绚丽而美好，带来时尚减...,类型#裙*版型#显瘦*风格#文艺*风格#简约*图案#印花*图案#撞色*裙下摆#压褶*裙长#连...,6.5,准确性：包含绝大多数关键词，但缺少'撞色'（虽然提到'撞色镶边'，但未明确突出撞色）和'文艺...
9,清新的蓝色衣身搭配上可爱的荷叶边袖摆和蝴蝶结胸饰更是让整个服装灵动起来，甜美俏皮气息尽显，让...,裙身处采用立体蝴蝶结装饰辅以蓝色条带点缀，令衣身造型饱满富有层次的同时为其注入一丝甜美气息。...,类型#裙*颜色#蓝色*风格#清新*图案#蝴蝶结,7.5,文案准确覆盖了蓝色、清新、蝴蝶结关键词，但未包含立体蝴蝶结装饰和蓝色条带点缀，而是增加了荷叶...


In [36]:
llm_judge_data_df.describe()

,llm_judge_score
count,1051.000000
mean,6.546051
std,1.227530
min,0.000000
25%,6.000000
50%,6.500000
75%,7.500000
max,9.000000
